In [1]:
import os
import json
import time
import warnings

from dotenv import load_dotenv
from github import Github
from github.GithubException import RateLimitExceededException, GithubException

warnings.filterwarnings("ignore")


## Authenticate

In [2]:
load_dotenv()

TOKEN = os.getenv("GITHUB_TOKEN")
if not TOKEN:
    raise ValueError(
        "GITHUB_TOKEN not found. Add GITHUB_TOKEN=<your_token> to your .env file."
    )

g = Github(TOKEN)
rate_limit = g.get_rate_limit()
core = getattr(rate_limit, "resources", rate_limit).core
print(f"Authenticated. Rate limit: {core.remaining}/{core.limit} "
      f"(resets at {core.reset.isoformat()})")

Authenticated. Rate limit: 4677/5000 (resets at 2026-07-31T21:49:51+00:00)


## Helper: handle rate limits gracefully

GitHub's REST API has a request rate limit. This helper pauses and retries automatically when the limit is hit, and retries transient errors instead of killing the whole run.


In [3]:
def safe_call(fn, *args, max_retries=3, **kwargs):
    """Call a PyGithub function, retrying automatically on rate limit
    or transient errors. Raises after `max_retries` non-rate-limit failures.
    """
    attempts = 0
    while True:
        try:
            return fn(*args, **kwargs)
        except RateLimitExceededException:
            reset_time = g.get_rate_limit().core.reset.timestamp()
            sleep_for = max(reset_time - time.time(), 1) + 5
            print(f"Rate limit hit. Sleeping for {sleep_for:.0f} seconds...")
            time.sleep(sleep_for)
        except GithubException as e:
            attempts += 1
            if attempts > max_retries:
                print(f"GitHub API error after {max_retries} retries: {e}")
                raise
            wait = 2 ** attempts
            print(f"GitHub API error ({e}). Retrying in {wait}s "
                  f"({attempts}/{max_retries})...")
            time.sleep(wait)


## Collecting PR data

For each Pull Request, this collects:

- **Metadata** — title, description, author, branches, SHAs, labels, stats
- **Files** — per-file diffs/patches (with status, additions, deletions)
- **Commits** — SHA, message, author, date
- **Discussion comments** — general conversation comments on the PR
- **Review comments** — inline code review comments tied to diff lines
- **Reviews** — top-level review summaries (APPROVED / CHANGES_REQUESTED / COMMENTED)

In [4]:
def collect_pr_data(repo, state="closed", max_prs=50):
    """
    Collect GitHub Pull Request data (metadata, diffs, comments, reviews)
    for up to `max_prs` pull requests from `repo`.

    Returns a list of dicts, one per PR.
    """
    results = []
    pulls = safe_call(repo.get_pulls, state=state, sort="updated", direction="desc")

    for idx, pr in enumerate(pulls):
        if idx >= max_prs:
            break

        print(f"[{idx + 1}/{max_prs}] PR #{pr.number}: {pr.title[:70]}")

        try:
            pr_record = {
                "repository": repo.full_name,
                "repository_url": repo.html_url,
                "repository_language": repo.language,

                "pr_number": pr.number,
                "title": pr.title,
                "description": pr.body,
                "url": pr.html_url,

                "state": pr.state,
                "merged": pr.merged,
                "author": pr.user.login if pr.user else None,

                "created_at": pr.created_at.isoformat() if pr.created_at else None,
                "updated_at": pr.updated_at.isoformat() if pr.updated_at else None,
                "closed_at": pr.closed_at.isoformat() if pr.closed_at else None,

                "base_branch": pr.base.ref,
                "head_branch": pr.head.ref,
                "base_sha": pr.base.sha,
                "head_sha": pr.head.sha,
                "merge_commit_sha": pr.merge_commit_sha,

                "changed_files": pr.changed_files,
                "commits": pr.commits,
                "additions": pr.additions,
                "deletions": pr.deletions,

                "labels": [label.name for label in pr.labels],

                "files": [],
                "discussion_comments": [],
                "review_comments": [],
                "reviews": [],
                "commits_data": [],
            }

            for file in safe_call(pr.get_files):
                if file.patch is None:
                    continue
                pr_record["files"].append({
                    "filename": file.filename,
                    "status": file.status,
                    "additions": file.additions,
                    "deletions": file.deletions,
                    "changes": file.changes,
                    "patch": file.patch,
                    "blob_url": file.blob_url,
                    "raw_url": file.raw_url,
                })

            for commit in safe_call(pr.get_commits):
                pr_record["commits_data"].append({
                    "sha": commit.sha,
                    "message": commit.commit.message,
                    "author": commit.author.login if commit.author else None,
                    "date": (
                        commit.commit.author.date.isoformat()
                        if commit.commit.author else None
                    ),
                })

            for comment in safe_call(pr.get_issue_comments):
                pr_record["discussion_comments"].append({
                    "user": comment.user.login if comment.user else None,
                    "body": comment.body,
                    "created_at": (
                        comment.created_at.isoformat() if comment.created_at else None
                    ),
                    "url": comment.html_url,
                })

            for comment in safe_call(pr.get_review_comments):
                pr_record["review_comments"].append({
                    "user": comment.user.login if comment.user else None,
                    "body": comment.body,
                    "path": comment.path,
                    "line": comment.line,
                    "commit_id": comment.commit_id,
                    "created_at": (
                        comment.created_at.isoformat() if comment.created_at else None
                    ),
                    "url": comment.html_url,
                })

            for review in safe_call(pr.get_reviews):
                pr_record["reviews"].append({
                    "user": review.user.login if review.user else None,
                    "state": review.state,
                    "body": review.body,
                    "submitted_at": (
                        review.submitted_at.isoformat() if review.submitted_at else None
                    ),
                })

            pr_record["statistics"] = {
                "num_files": len(pr_record["files"]),
                "num_discussion_comments": len(pr_record["discussion_comments"]),
                "num_review_comments": len(pr_record["review_comments"]),
                "num_reviews": len(pr_record["reviews"]),
                "num_commits": len(pr_record["commits_data"]),
            }

            results.append(pr_record)

        except GithubException as e:
            print(f"  Skipping PR #{pr.number} due to error: {e}")
            continue

    return results


## Save records to disk

In [5]:
def save_records(records, repo_full_name, output_dir="data"):
    """Save PR records as JSON Lines (one JSON object per line)."""
    os.makedirs(output_dir, exist_ok=True)
    repo_slug = repo_full_name.replace("/", "_")
    output_path = os.path.join(output_dir, f"{repo_slug}_pr_data.jsonl")

    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Saved {len(records)} PRs -> {output_path}")
    return output_path


## Single-repo run

Test the pipeline on one repository before scaling up.

In [6]:
REPO_NAME = "django/django"

repo = g.get_repo(REPO_NAME)
print(repo.full_name, "-", repo.description)


django/django - The Web framework for perfectionists with deadlines.


In [7]:
data = collect_pr_data(repo, state="closed", max_prs=20)
print(f"Collected data for {len(data)} pull requests.")


[1/20] PR #21310: Fixed #36626 -- geodjango -- Preserved milliseconds when parsing GPX …
[2/20] PR #21709: Fixed #37236 -- Allowed altering spatial indexes on RasterField.
[3/20] PR #21629: Fixed #27734 -- Made parallel test workers reuse database clones of ex
[4/20] PR #21706: Removed advice to include ticket numbers in tests.
[5/20] PR #21693: Fixed #37238 -- Fixed unintentional fallback to python default for a d
[6/20] PR #18506: Fixes [#25656] Stop displaying links in recent changes when user does 
[7/20] PR #21699: Fixed #37235 -- Added compatibility for sqlparse 0.5.5.
[8/20] PR #21700: Fixed #37240 -- Fixed simplify_regex() with multiple unnamed groups.
[9/20] PR #21704: Updated asgiref dependency in free-threaded requirements.
[10/20] PR #21703: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and Cross-Origin-Re
[11/20] PR #21701: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and Cross-Origin-Re
[12/20] PR #21702: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and C

In [8]:
save_records(data, repo.full_name)


Saved 20 PRs -> data\django_django_pr_data.jsonl


'data\\django_django_pr_data.jsonl'

In [9]:
if data:
    sample = data[0]
    print("Sample PR:", sample["title"])
    print("Files:", sample["statistics"]["num_files"])
    print("Discussion comments:", sample["statistics"]["num_discussion_comments"])
    print("Review comments:", sample["statistics"]["num_review_comments"])
    print("Reviews:", sample["statistics"]["num_reviews"])


Sample PR: Fixed #36626 -- geodjango -- Preserved milliseconds when parsing GPX …
Files: 6
Discussion comments: 7
Review comments: 2
Reviews: 4


## Multi-repo run

Collects the same data across many repositories spanning several languages.


In [10]:
REPOSITORIES = [
    # Python
    "django/django",
    "fastapi/fastapi",
    "pallets/flask",
    "pydantic/pydantic",
    # Java
    "spring-projects/spring-framework",
    "apache/kafka",
    # JavaScript / TypeScript
    "microsoft/vscode",
    "facebook/react",
    "vercel/next.js",
    # Go
    "kubernetes/kubernetes",
    "gin-gonic/gin",
    # C++
    "opencv/opencv",
    # Machine Learning
    "tensorflow/tensorflow",
    "pytorch/pytorch",
]

MAX_PRS_PER_REPO = 10
OUTPUT_DIR = "data"

total_prs = 0
failed_repos = []

for repo_name in REPOSITORIES:
    print("\n" + "=" * 80)
    print(f"Collecting: {repo_name}")
    print("=" * 80)

    try:
        current_repo = g.get_repo(repo_name)
        repo_data = collect_pr_data(current_repo, state="closed", max_prs=MAX_PRS_PER_REPO)
        save_records(repo_data, current_repo.full_name, output_dir=OUTPUT_DIR)
        total_prs += len(repo_data)

    except Exception as e:
        print(f"Failed: {repo_name} -> {e}")
        failed_repos.append(repo_name)

print("\n" + "=" * 80)
print(f"Finished collecting {total_prs} Pull Requests.")
print(f"Datasets saved in: {OUTPUT_DIR}")
if failed_repos:
    print(f"Failed repos ({len(failed_repos)}): {failed_repos}")
print("=" * 80)



Collecting: django/django
[1/10] PR #21310: Fixed #36626 -- geodjango -- Preserved milliseconds when parsing GPX …
[2/10] PR #21709: Fixed #37236 -- Allowed altering spatial indexes on RasterField.
[3/10] PR #21629: Fixed #27734 -- Made parallel test workers reuse database clones of ex
[4/10] PR #21706: Removed advice to include ticket numbers in tests.
[5/10] PR #21693: Fixed #37238 -- Fixed unintentional fallback to python default for a d
[6/10] PR #18506: Fixes [#25656] Stop displaying links in recent changes when user does 
[7/10] PR #21699: Fixed #37235 -- Added compatibility for sqlparse 0.5.5.
[8/10] PR #21700: Fixed #37240 -- Fixed simplify_regex() with multiple unnamed groups.
[9/10] PR #21704: Updated asgiref dependency in free-threaded requirements.
[10/10] PR #21703: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and Cross-Origin-Re
Saved 10 PRs -> data\django_django_pr_data.jsonl

Collecting: fastapi/fastapi
[1/10] PR #16087: ♻️ Replace list with a set for visited cac

Following Github server redirection from /repos/facebook/react to /repositories/10270250


[1/10] PR #37063: [Fiber] Collect Host Singleton children of Fragments
[2/10] PR #37154: [Flight] Add 'pending_weak' to Flight thenable protocol
[3/10] PR #36269: [compiler] Fix false positive for local mutation in filter callbacks
[4/10] PR #37104: [Fiber] Warn for Conditional Use of use() Based on Cache
[5/10] PR #35198: Improve TypeScript type definitions in jest.d.ts
[6/10] PR #36337: Fix false positive for self-referential useCallback
[7/10] PR #37152: [DevTools] Remove FlowFixMe from extension lifecycle
[8/10] PR #37151: [DevTools] Create extension panels before React detection
[9/10] PR #37155: [DevTools] Reset extension backend on pagehide
[10/10] PR #37143: Add ReactDOM `browser()` API
Saved 10 PRs -> data\react_react_pr_data.jsonl

Collecting: vercel/next.js
[1/10] PR #96442: [react-sync] Enable auto-merge on PRs
[2/10] PR #96438: [ci] Run new/changed deploy tests asap
[3/10] PR #96181: Emit static hints for partial prerenders
[4/10] PR #96434: Upgrade React from `0f42eac2-20